In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2
from psycopg2.extras import execute_batch
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from dotenv import load_dotenv


# Análisis Exploratorio de Datos (EDA) — Rendimiento Técnico-Táctico del Plantel

## Objetivos del Análisis
Este cuaderno explora las métricas individuales y colectivas del equipo para identificar:
- Los principales perfiles generadores de gol (goleadores, asistidores y contribuciones totales).
- La relación entre el tiempo de juego (`pct_titularidad`) y la efectividad de cara a puerta.
- Patrones de eficiencia ofensiva que ayuden a la toma de decisiones del cuerpo técnico.

In [33]:
load_dotenv()

database_url = os.getenv("DATABASE_URL")


query = "SELECT * FROM jugadores_stats;"

with psycopg2.connect(database_url, client_encoding='utf8') as conn:
    df = pd.read_sql_query(query, conn)
df

C:\Users\aabap\AppData\Local\Temp\ipykernel_51124\2202220803.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,id,nombre,partido,titular,suplente,goles,asist,posicion,gol_x_part,asist_x_part,contribucion_gol,pct_partidos_jugados,pct_titularidad
0,1,Jorge Rodriguez,25,17,8,0,0,Port,0.00,0.00,0.00,83.3,68.0
1,2,Javier Sanchez,27,14,13,0,0,Port,0.00,0.00,0.00,90.0,51.9
2,3,Manuel Hermoso,25,19,6,0,0,Cent,0.00,0.00,0.00,83.3,76.0
3,4,Martin Peiro,22,6,16,0,0,Cent,0.00,0.00,0.00,73.3,27.3
4,5,Miguel Gutierrez,29,28,1,7,2,Cent,0.24,0.07,0.31,96.7,96.6
5,6,Kosta Velazco,25,19,6,5,6,Lat,0.20,0.24,0.44,83.3,76.0
6,7,Jaime Cerezo,27,24,3,4,7,Lat,0.15,0.26,0.41,90.0,88.9
7,8,Alejandro Martinez,21,5,16,2,5,Lat,0.10,0.24,0.33,70.0,23.8
8,9,Pablo Leon,19,14,5,1,3,Lat,0.05,0.16,0.21,63.3,73.7
9,10,Hugo Mauro,25,11,14,1,0,Lat,0.04,0.00,0.04,83.3,44.0


## 1. Perfiles de Impacto Ofensivo Directo

Analizamos a los jugadores con mayor participación directa en el marcador mediante tres métricas clave:
1. **Goles anotados**
2. **Asistencias repartidas**
3. **Contribuciones directas de gol** ($\text{Goles} + \text{Asistencias}$)

In [34]:
top_goles = df.nlargest(5, 'goles').sort_values('goles')

fig = px.bar(
    top_goles,
    x='goles',
    y='nombre',
    orientation='h',                             
    title="Top 5 Goleadores",
    labels={'goles': 'Goles', 'nombre': 'Jugador'},
    color_discrete_sequence=['#426a09'],        
    text='goles'                                 
)

fig.update_layout(
    xaxis_title="Goles",
    yaxis_title="Jugador",
    height=400
)

fig.update_traces(textposition='outside')

fig.show(renderer="vscode")  

In [35]:
top_asist = df.nlargest(5, 'asist').sort_values('asist')

fig = px.bar(
    top_asist,
    x='asist',
    y='nombre',
    orientation='h',                             
    title="Top 5 Asistidores",
    labels={'asist': 'Asistencias', 'nombre': 'Jugador'},
    color_discrete_sequence=["#090f6a"],         
    text='asist' 
)                                

fig.update_layout(
    xaxis_title="Asistencias",
    yaxis_title="Jugador",
    height=400
)

fig.update_traces(textposition='outside')

fig.show(renderer="vscode") 

In [43]:
top_contribucion_gol = df.nlargest(5, 'contribucion_gol').sort_values('contribucion_gol')

fig = px.bar(
    top_contribucion_gol,
    x='contribucion_gol',
    y='nombre',
    orientation='h',                             
    title="Top 5 Contribuciones de gol X partido",
    labels={'contribucion_gol': 'PCT Contribuciones de gol', 'nombre': 'Jugador'},
    color_discrete_sequence=["#bfc20d"],         
    text='contribucion_gol'
)                                

fig.update_layout(
    xaxis_title="Contribucion de gol X partido",
    yaxis_title="Jugador",
    height=400
)

fig.update_traces(textposition='outside')

fig.show(renderer="vscode") 

> **Insights Clave de Generación Ofensiva:**
> - **Concentración del gol:** Un grupo reducido de atacantes acumula la mayor parte del volumen goleador del equipo.
> - **Generadores de juego:** Alejandro Galindo, Diego Baptista y Hugo Díaz destacan como los pilares del ataque, no solo finalizando jugadas sino repartiendo asistencias clave.

## 2. Relación entre Titularidad y Producción Ofensiva

Analizamos si una mayor presencia en el XI inicial (`pct_titularidad`) se traduce proporcionalmente en una mayor aportación de gol (`contribucion_gol`). 

Las líneas discontinuas representan la **media del equipo**, dividiendo el gráfico en 4 cuadrantes analíticos:
- **Cuadrante Superior Derecho:** Jugadores titulares e indiscutibles de alto impacto.
- **Cuadrante Superior Izquierdo:** *Revulsivos eficaces* (alto rendimiento en menos minutos).

In [37]:
fig1 = px.scatter(
    df, 
    x='pct_titularidad', 
    y='contribucion_gol',
    color='posicion',
    hover_name='nombre',                  # Muestra el nombre al pasar el cursor
    hover_data=['goles', 'asist'],        # Muestra datos extra en la ventana emergente
    labels={'pct_titularidad': '% Titularidad', 'produccion_x_part': 'Producción / Partido'},
    title="Matriz de Eficiencia (Interactiva)"
)

# 3. Líneas de referencia (Promedios)
fig1.add_vline(x=df['pct_titularidad'].mean(), line_dash="dash", line_color="gray")
fig1.add_hline(y=df['contribucion_gol'].mean(), line_dash="dash", line_color="gray")

fig1.show(renderer="vscode")

> **Insights Clave de la Distribución:**
> - Se observa una correlación positiva clara entre los minutos disputados de titular y la producción global de goles.
> - Los jugadores ubicados por encima de la media horizontal muestran una eficiencia destacada respecto al promedio de la plantilla.

## 3. Análisis de Pareto: Concentración del Gol

Aplicamos el principio de Pareto (regla 80/20) para evaluar la **dependencia ofensiva del equipo**. 

Este análisis combina:
- **Barras:** La contribución individual de gol de cada jugador.
- **Línea acumulada (roja):** El porcentaje acumulado del total de goles del equipo.
- **Línea referencia (80%):** Umbral para identificar el grupo reducido de jugadores que genera la gran mayoría de las ocasiones de gol.

In [38]:

df_ordenado = df.sort_values(by='contribucion_gol', ascending=False)
total_goles = df_ordenado['contribucion_gol'].sum()
df_ordenado['pct_acumulado'] = (df_ordenado['contribucion_gol'].cumsum() / total_goles) * 100

fig2 = make_subplots(specs=[[{"secondary_y": True}]])

fig2.add_trace(
    go.Bar(x=df_ordenado['nombre'], y=df_ordenado['contribucion_gol'], name="Contribución Gol"),
    secondary_y=False
)

fig2.add_trace(
    go.Scatter(x=df_ordenado['nombre'], y=df_ordenado['pct_acumulado'], name="% Acumulado", mode='lines+markers', line=dict(color='red')),
    secondary_y=True
)

fig2.add_hline(y=80, line_dash="dash", line_color="gray", secondary_y=True)

fig2.update_layout(title="Concentración del Gol - Pareto (Interactivo)")
fig2.update_yaxes(title_text="Contribución Total", secondary_y=False)
fig2.update_yaxes(title_text="% Acumulado", range=[0, 105], secondary_y=True)

fig2.show(renderer="vscode")  

> **Insights Clave de Concentración (Pareto):**
> - **Núcleo ofensivo:** El 80% de las contribuciones de gol está concentrado en un porcentaje reducido de la plantilla.
> - **Riesgo por dependencia:** Permite identificar la vulnerabilidad del equipo ante posibles bajas (lesiones o sanciones) de los jugadores clave situados a la izquierda de la línea del 80%.

In [39]:
metricas = ['gol_x_part', 'asist_x_part', 'contribucion_gol', 'pct_titularidad']
etiquetas = ['Goles / Partido', 'Asistencias / Partido', 'Contribución Gol', '% Titularidad']

df_percentiles = df[metricas].rank(pct=True) * 100

#  Elegir un jugador por su índice (ejemplo: primer jugador df.iloc[0])
idx = 13
nombre_jugador = df.iloc[idx]['nombre']
valores_pct = df_percentiles.iloc[idx].values
valores_reales = df.iloc[idx][metricas].values  # Para mostrar el dato real en el hover

fig4 = go.Figure()

for i in range(len(metricas)):
    fig4.add_trace(go.Scatter(
        x=[0, valores_pct[i]],
        y=[etiquetas[i], etiquetas[i]],
        mode='lines',
        line=dict(color='#2b5c8f', width=4),
        showlegend=False,
        hoverinfo='skip'
    ))

fig4.add_trace(go.Scatter(
    x=valores_pct,
    y=etiquetas,
    mode='markers+text',
    marker=dict(color='#2b5c8f', size=16),
    text=[f"{v:.0f}%" for v in valores_pct],
    textposition="top center",
    hovertemplate="<b>%{y}</b><br>Percentil: %{x:.1f}%<br>Dato Real: %{customdata}<extra></extra>",
    customdata=valores_reales,
    name=nombre_jugador
))

# Línea vertical del Percentil 50 (Media del equipo)
fig4.add_vline(x=50, line_dash="dash", line_color="gray", annotation_text="Media del equipo (P50)")

# Formato y rangos
fig4.update_layout(
    title=f"Perfil de Impacto en Percentiles: <b>{nombre_jugador}</b>",
    xaxis=dict(title="Percentil en la Plantilla (0 = Mínimo, 100 = Máximo)", range=[0, 105]),
    yaxis=dict(title="Métricas"),
    showlegend=False
)

fig4.show(renderer="vscode")  # o renderer="browser"

## 4. Distribución de la Producción Ofensiva por Posición

Analizamos cómo se distribuye la contribución de gol ($\text{Goles} + \text{Asistencias}$) según la posición táctica en el campo.

El gráfico de caja (boxplot) con todos los puntos visibles (`points='all'`) permite observar:
- La mediana y dispersión del rendimiento dentro de cada demarcación.
- Los **jugadores atípicos o destacados** (*outliers*) que aportan un volumen de gol superior al promedio de su línea táctica.

In [40]:
fig3 = px.box(
    df,
    x='posicion',
    y='contribucion_gol',
    color='posicion',
    points='all',                  
    hover_name='nombre',            
    hover_data=['goles', 'asist'],  
    title="Distribución de la Contribución a Gol por Posición"
)

fig3.update_layout(
    xaxis_title="Posición",
    yaxis_title="Goles + Asistencias Totales",
    showlegend=False
)

fig3.show(renderer="vscode")  

> **Insights Clave por Posición:**
> - **Línea de ataque vs. resto:** Como es esperable, la delantera concentra los valores más altos de contribución directas de gol.
> - **Aportación de segunda línea:** Permite identificar la influencia ofensiva de mediocampistas y laterales en la construcción del juego del equipo.
> - **Variabilidad interna:** Las diferencias en el tamaño de las cajas muestran qué posiciones tienen un rendimiento ofensivo homogénico y cuáles dependen de actuaciones individuales concretas.

## Conclusiones Globales del EDA

1. **Concentración en la producción:** Existe una alta dependencia de un núcleo de 3-5 jugadores clave para la generación y definición de jugadas de gol.
2. **Rendimiento según rol:** Los datos muestran una correlación directa entre el volumen de minutos jugados como titular y la aportación anotadora.
3. **Optimización de recursos:** El cuerpo técnico puede utilizar estos perfiles para planificar rotaciones sin perder la eficiencia ofensiva identificada en los revulsivos.